In [12]:
import fitz  # PyMuPDF
import string

# Corrected dictionary with your absolute paths
figure_list = {
    '1A': '/home3/oml4h/PLM_SARS-CoV-2/Results/mut_access_final_figures/clade_viz_px445235_relabelled_evenbiggerfont.pdf',
    '1B&C': '/home3/oml4h/PLM_SARS-CoV-2/Sequences/IAV_lineage_files/IAV_model_comparison_aa/codon_to_amino_acid_matrix_heatmap.pdf',
    '1D': '/home3/oml4h/PLM_SARS-CoV-2/Sequences/IAV_lineage_files/IAV_model_comparison_aa/codon_to_amino_acid_matrix_heatmap.pdf',
    #'2A': '/home3/oml4h/PLM_SARS-CoV-2/Results/iav_mutational_accessibility/Lytras_OG/plots/per_model/ESM2_650M_HA80_raw/plm_vs_mut_prob_scatter.pdf',
    #'2B': '/home3/oml4h/PLM_SARS-CoV-2/Results/iav_mutational_accessibility/Lytras_OG/plots/per_model/ESM2_650M_HA80/plm_vs_mut_prob_scatter.pdf',
    '2A': '/home3/oml4h/PLM_SARS-CoV-2/Results/iav_mutational_accessibility/Lytras_OG/plots/per_model/ESM2_650M_HA80_raw/plm_vs_mut_prob_scatter_by_mutation_distance.pdf',
    '2B': '/home3/oml4h/PLM_SARS-CoV-2/Results/iav_mutational_accessibility/Lytras_OG/plots/per_model/ESM2_650M_HA80/plm_vs_mut_prob_scatter_by_mutation_distance.pdf',
    '2C': '/home3/oml4h/PLM_SARS-CoV-2/Results/mut_access_final_figures/plot_PLM_epoch.png',
    '3': '/home3/oml4h/PLM_SARS-CoV-2/Results/iav_mutational_accessibility/Lytras_OG/plots/per_model/ESM2_650M_HA80/alpha_sweep_metrics_selected_with_raw.pdf'
}

def generate_composite_figure(fig_keys, layout, output_filename, font_size=28, offset=20):
    """
    Compiles specific panels into a single multi-panel figure.
    
    :param fig_keys: List of keys from figure_list (e.g., ['1A', '1B&C', '1D'])
    :param layout: Tuple indicating (rows, cols)
    :param output_filename: Destination path for the output PDF
    """
    rows, cols = layout
    doc_out = fitz.open()
    
    # Open inputs and sample sizes
    opened_docs = {}
    sample_w, sample_h = 600, 500  # Default fallback if sizes vary wildly
    
    for key in fig_keys:
        path = figure_list[key]
        if path.lower().endswith('.pdf'):
            d = fitz.open(path)
            opened_docs[key] = d
            # Use the first valid PDF to anchor standard panel dimensions
            sample_w = d[0].rect.width
            sample_h = d[0].rect.height
        else:
            opened_docs[key] = path  # Save path string for images
    font_path = "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"
    # Create master canvas
    master_page = doc_out.new_page(width=sample_w * cols, height=sample_h * rows)
    
    idx = 0
    for r in range(rows):
        for c in range(cols):
            if idx >= len(fig_keys):
                break
                
            key = fig_keys[idx]
            source = opened_docs[key]
            
            # Map target bounding box
            x0, y0 = c * sample_w, r * sample_h
            x1, y1 = x0 + sample_w, y0 + sample_h
            target_rect = fitz.Rect(x0, y0, x1, y1)
            
            # Handle PDF vs PNG source placement
            if isinstance(source, fitz.Document):
                master_page.show_pdf_page(target_rect, source, 0)
            else:
                # Source is a PNG path string; open as pixmap and insert
                img_doc = fitz.open(source)
                pix = img_doc[0].get_pixmap()
                master_page.insert_image(target_rect, pixmap=pix)
                img_doc.close()
                
            # Clean letter assignment (extract suffix: '1A' -> 'A', '1B&C' -> 'B')
            # If it's a standalone figure like '3', just use 'A' or the key itself
            raw_letter = ''.join([char for char in key if char.isalpha()])
            display_letter = raw_letter[0] if raw_letter else "A"
            
            # Stamp the letter label
            text_point = fitz.Point(x0 + offset, y0 + offset + font_size)
            master_page.insert_text(
                text_point, 
                display_letter, 
                fontsize=font_size, 
                fontname="liberation-bold", # A unique identifier string for this run
                fontfile=font_path,         # Explicit path to the binary file
                color=(0, 0, 0)
            )
            idx += 1
            
    doc_out.save(output_filename)
    doc_out.close()
    
    # Close opened PDF documents
    for doc in opened_docs.values():
        if isinstance(doc, fitz.Document):
            doc.close()
            
    print(f"Successfully generated: {output_filename}")


# --- Execution Pipeline ---

# Figure 1: 3 panels arranged in 1 Row, 3 Columns (or 3 Rows, 1 Column depending on your preference)
generate_composite_figure(
    fig_keys=['1A', '1B&C', '1D'], 
    layout=(1, 3), 
    output_filename='/home3/oml4h/PLM_SARS-CoV-2/Results/mut_access_final_figures/panelstrim2/Figure_1_Composite.pdf'
)

# Figure 2: 3 panels (2 PDFs, 1 PNG) arranged in a 2x2 grid (leaving the 4th slot blank)
generate_composite_figure(
    fig_keys=['2A', '2B', '2C'], 
    layout=(3, 1), 
    output_filename='/home3/oml4h/PLM_SARS-CoV-2/Results/mut_access_final_figures/panels2/Figure_2_Composite.pdf'
)
generate_composite_figure(
    fig_keys=['2A', '2B', '2C'], 
    layout=(2, 2), 
    output_filename='/home3/oml4h/PLM_SARS-CoV-2/Results/mut_access_final_figures/panels2/Figure_2_Composite22.pdf'
)

# Figure 3: Single standalone panel (No merging required, but adds the label 'A')
generate_composite_figure(
    fig_keys=['3'], 
    layout=(1, 1), 
    output_filename='/home3/oml4h/PLM_SARS-CoV-2/Results/mut_access_final_figures/panels2/Figure_3_Composite.pdf'
)

Successfully generated: /home3/oml4h/PLM_SARS-CoV-2/Results/mut_access_final_figures/panelstrim2/Figure_1_Composite.pdf
Successfully generated: /home3/oml4h/PLM_SARS-CoV-2/Results/mut_access_final_figures/panels2/Figure_2_Composite.pdf
Successfully generated: /home3/oml4h/PLM_SARS-CoV-2/Results/mut_access_final_figures/panels2/Figure_2_Composite22.pdf
Successfully generated: /home3/oml4h/PLM_SARS-CoV-2/Results/mut_access_final_figures/panels2/Figure_3_Composite.pdf


In [8]:
import fitz  # PyMuPDF

def generate_dynamic_row_figure(fig_keys, output_filename, font_scale=0.04, default_offset=20, custom_labels=None):
    """
    Compiles specific panels into a single vertical column.
    
    :param font_scale: Font size as a fraction of total panel width
    :param default_offset: Default internal padding for automatic labels
    :param custom_labels: Dict for manual text overrides. 
                          Format: {'key': [("Text", x, y), ("Text2", x2, y2)]}
                          Coordinates (x, y) are relative to that specific panel's top-left corner.
    """
    if custom_labels is None:
        custom_labels = {}
        
    doc_out = fitz.open()
    opened_docs = []
    total_height = 0
    max_width = 0
    
    # First pass: Open files and determine master canvas width
    for key in fig_keys:
        path = figure_list[key]
        if path.lower().endswith('.pdf'):
            d = fitz.open(path)
            w, h = d[0].rect.width, d[0].rect.height
            opened_docs.append((key, d, w, h))
        else:
            img_doc = fitz.open(path)
            pix = img_doc[0].get_pixmap()
            opened_docs.append((key, img_doc, pix.width, pix.height))
            
        if w > max_width:
            max_width = w

    # Proportional font sizing based on final canvas width
    font_size = int(max_width * font_scale)
    
    # Second pass: Standardise widths and scale heights proportionally
    panel_specs = []
    for key, doc, w, h in opened_docs:
        scale_factor = max_width / w
        scaled_h = h * scale_factor
        panel_specs.append((key, doc, max_width, scaled_h))
        total_height += scaled_h

    # Create the master canvas
    master_page = doc_out.new_page(width=max_width, height=total_height)
    
    # Third pass: Place panels and layer labels
    current_y = 0
    font_path = "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"
    
    for key, source, w, h in panel_specs:
        target_rect = fitz.Rect(0, current_y, w, current_y + h)
        
        # Place graphic content
        if isinstance(source, fitz.Document) and source.name.lower().endswith('.pdf'):
            master_page.show_pdf_page(target_rect, source, 0)
        else:
            pix = source[0].get_pixmap()
            master_page.insert_image(target_rect, pixmap=pix)
            
        # Check if this panel has explicit custom coordinate labels defined
        if key in custom_labels:
            for label_text, rel_x, rel_y in custom_labels[key]:
                # Map relative panel coordinates to absolute master page coordinates
                abs_x = rel_x
                abs_y = current_y + rel_y
                text_point = fitz.Point(abs_x, abs_y)
                
                master_page.insert_text(
                    text_point, 
                    label_text, 
                    fontsize=font_size, 
                    fontname="liberation-bold", 
                    fontfile=font_path,         
                    color=(0, 0, 0)
                )
        else:
            # Fall back to automatic generation loop if no custom override exists
            raw_letter = ''.join([char for char in key if char.isalpha()])
            display_letter = raw_letter[0] if raw_letter else "A"
            
            text_point = fitz.Point(default_offset, current_y + default_offset + font_size)
            master_page.insert_text(
                text_point, 
                display_letter, 
                fontsize=font_size, 
                fontname="liberation-bold", 
                fontfile=font_path,         
                color=(0, 0, 0)
            )
        
        current_y += h

    doc_out.save(output_filename)
    doc_out.close()
    
    for _, doc, _, _ in opened_docs:
        doc.close()
            
    print(f"Successfully generated: {output_filename}")

In [9]:
# Define manual coordinate maps for your labels
# Format: { 'panel_key': [ ("Text", X, Y), ("Text2", X, Y) ] }
# Note: PDF text coordinates track from the baseline (bottom) of the text strings.
my_custom_labels = {
    '1B&C': [
        ("B", 20, 45),    # Places 'B' near the top left of the second row
        ("C", 400, 45)   # Places 'C' shifted horizontally 400 points to the right
    ]
}

generate_dynamic_row_figure(
    fig_keys=['1A', '1B&C', '1D'], 
    default_offset=20,  # Used for panels without overrides (like 1A and 1D)
    custom_labels=my_custom_labels,
    output_filename='/home3/oml4h/PLM_SARS-CoV-2/Results/mut_access_final_figures/panelstrim2/Figure_1_Composite.pdf'
)

Successfully generated: /home3/oml4h/PLM_SARS-CoV-2/Results/mut_access_final_figures/panelstrim2/Figure_1_Composite.pdf


In [18]:
import fitz  # PyMuPDF

def generate_figure_2_asymmetric(fig_keys, output_filename, font_scale=0.04, offset=20, row2_scale=0.7, bottom_whitespace=300):
    """
    Compiles precisely 3 keys into a 2-row layout:
    Row 1: Two columns (Key 1 and Key 2 side-by-side, taking up full width)
    Row 2: One column (Key 3 shrunk down, flushed to top, extra padding dumped at the very bottom)
    
    :param row2_scale: Scaling factor for the bottom plot width (e.g., 0.7 = 70% of canvas width)
    :param bottom_whitespace: Number of points of blank canvas to append to the bottom of the PDF
    """
    if len(fig_keys) != 3:
        raise ValueError("This specific layout function expects exactly 3 figure keys (e.g., ['2A', '2B', '2C'])")

    doc_out = fitz.open()
    opened_docs = {}
    raw_dims = {}

    # Open files and get baseline dimensions
    for key in fig_keys:
        path = figure_list[key]
        if path.lower().endswith('.pdf'):
            d = fitz.open(path)
            opened_docs[key] = d
            raw_dims[key] = (d[0].rect.width, d[0].rect.height)
        else:
            img_doc = fitz.open(path)
            opened_docs[key] = img_doc
            pix = img_doc[0].get_pixmap()
            raw_dims[key] = (pix.width, pix.height)

    # Calculate master canvas layout based on aspect ratios
    half_w = max(raw_dims[fig_keys[0]][0], raw_dims[fig_keys[1]][0])
    master_width = half_w * 2

    # Scale Row 1 heights to match their new half_w box proportionally
    scale_1a = half_w / raw_dims[fig_keys[0]][0]
    scale_1b = half_w / raw_dims[fig_keys[1]][0]
    row1_h = max(raw_dims[fig_keys[0]][1] * scale_1a, raw_dims[fig_keys[1]][1] * scale_1b)

    # Calculate the scaled dimensions for Row 2 (Key 3)
    full_scale_2c = master_width / raw_dims[fig_keys[2]][0]
    shrunk_w2c = master_width * row2_scale
    shrunk_h2c = (raw_dims[fig_keys[2]][1] * full_scale_2c) * row2_scale
    
    # Total height incorporates the plots packed tightly + the bottom whitespace tail
    total_height = row1_h + shrunk_h2c + bottom_whitespace

    # Create master canvas page
    master_page = doc_out.new_page(width=master_width, height=total_height)
    font_path = "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"
    
    # Proportional font configuration based on final wide canvas
    font_size = int(master_width * font_scale)

    # --- Placement Step ---
    # Panel 1 (Top Left)
    rect_1 = fitz.Rect(0, 0, half_w, row1_h)
    # Panel 2 (Top Right)
    rect_2 = fitz.Rect(half_w, 0, master_width, row1_h)
    
    # Panel 3 (Bottom Row): Sized by row2_scale, centered horizontally,
    # but flushed vertically right against the bottom edge of Row 1 (row1_h)
    x0_row2 = (master_width - shrunk_w2c) / 2
    y0_row2 = row1_h  # Zero gap between rows
    x1_row2 = x0_row2 + shrunk_w2c
    y1_row2 = y0_row2 + shrunk_h2c
    rect_3 = fitz.Rect(x0_row2, y0_row2, x1_row2, y1_row2)

    targets = [rect_1, rect_2, rect_3]

    for idx, key in enumerate(fig_keys):
        source = opened_docs[key]
        target_rect = targets[idx]

        # Render Content
        if isinstance(source, fitz.Document) and source.name.lower().endswith('.pdf'):
            master_page.show_pdf_page(target_rect, source, 0)
        else:
            pix = source[0].get_pixmap()
            master_page.insert_image(target_rect, pixmap=pix)

        # Labels placement logic
        raw_letter = ''.join([char for char in key if char.isalpha()])
        display_letter = raw_letter[0] if raw_letter else "A"
        
        text_point = fitz.Point(target_rect.x0 + offset, target_rect.y0 + offset + font_size)
        master_page.insert_text(
            text_point, 
            display_letter, 
            fontsize=font_size, 
            fontname="liberation-bold", 
            fontfile=font_path,         
            color=(0, 0, 0)
        )

    doc_out.save(output_filename)
    doc_out.close()
    
    for doc in opened_docs.values():
        doc.close()
        
    print(f"Successfully generated layout with disposable bottom margin: {output_filename}")
    
generate_figure_2_asymmetric(
    fig_keys=['2A', '2B', '2C'],
    output_filename='/home3/oml4h/PLM_SARS-CoV-2/Results/mut_access_final_figures/panels2/Figure_2_Composite_Asymmetric.pdf',
    font_scale=0.04,
    offset=10,
    row2_scale=0.60,         # Shrinks the graphic cleanly
    bottom_whitespace=400   # Appends 400 points of empty canvas to the bottom zone
)

Successfully generated layout with disposable bottom margin: /home3/oml4h/PLM_SARS-CoV-2/Results/mut_access_final_figures/panels2/Figure_2_Composite_Asymmetric.pdf
